# iWildCam Results Analysis Notebook

This notebook is designed for post-run analysis of `iwildcam_iro` experiments produced by the standard `iro train` / `iro eval` JSONL artifacts.

What this notebook gives you:
- robust JSONL loading from nested results folders
- run health checks (failed/ok, seed coverage)
- algorithm-level summary tables (mean/std over seeds)
- validation metric visualizations (accuracy and macro-F1)
- training dynamics over steps from `result.history`
- optional IRO alpha-preference sweep analysis (if `iro eval` records exist)
- CSV + figure export for reports


In [ ]:
# Setup: imports, plotting defaults, and display options
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "seaborn", "-q"])
    import seaborn as sns

sns.set_theme(style="whitegrid", context="talk")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

RNG = np.random.default_rng(0)


## Configuration

Update paths if needed. The loader scans recursively for `*.jsonl`.


In [ ]:
# Configuration
RESULTS_ROOT_CANDIDATES = [
    Path("/home/c01josh/CISPA-scratch/c01josh/iro/runs/iwildcam_long/results"),
    Path("/home/c01josh/CISPA-scratch/c01josh/iro/runs/iwildcam_full/results"),
    Path("/home/c01josh/CISPA-scratch/c01josh/iro/iro_exp/results"),
]

RESULTS_ROOT = next((p for p in RESULTS_ROOT_CANDIDATES if p.exists()), RESULTS_ROOT_CANDIDATES[0])
DATASET_FILTER = "iwildcam"  # set to None to load all datasets
OUT_DIR = Path("/home/c01josh/CISPA-scratch/c01josh/iro/collected_results/iwildcam_analysis")
FIG_DIR = OUT_DIR / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"RESULTS_ROOT: {RESULTS_ROOT}")
print(f"RESULTS_ROOT exists: {RESULTS_ROOT.exists()}")
print(f"OUT_DIR: {OUT_DIR}")


## Loader Utilities


In [ ]:
def _is_scalar(v: Any) -> bool:
    return isinstance(v, (int, float, str, bool)) or v is None


def _to_float(v: Any) -> float | None:
    if v is None:
        return None
    if isinstance(v, (int, float)):
        return float(v)
    try:
        return float(v)
    except (TypeError, ValueError):
        return None


def iter_jsonl_paths(root: Path) -> list[Path]:
    if not root.exists():
        return []
    return sorted(root.rglob("*.jsonl"))


def load_iwildcam_records(results_root: Path, dataset_filter: str | None = "iwildcam"):
    run_rows: list[dict[str, Any]] = []
    history_rows: list[dict[str, Any]] = []
    split_rows: list[dict[str, Any]] = []
    eval_rows: list[dict[str, Any]] = []
    error_rows: list[dict[str, Any]] = []

    for path in iter_jsonl_paths(results_root):
        with path.open("r", encoding="utf-8") as f:
            for line_no, line in enumerate(f, start=1):
                line = line.strip()
                if not line:
                    continue
                try:
                    rec = json.loads(line)
                except json.JSONDecodeError:
                    error_rows.append({
                        "file": str(path),
                        "line": line_no,
                        "error_type": "JSONDecodeError",
                        "error_message": "Could not parse JSON line",
                    })
                    continue

                dataset_name = rec.get("dataset_name")
                if dataset_filter is not None and dataset_name != dataset_filter:
                    continue

                result = rec.get("result") or {}

                run = {
                    "file": str(path),
                    "line": line_no,
                    "run_id": rec.get("run_id"),
                    "status": rec.get("status"),
                    "experiment": rec.get("experiment"),
                    "exp_name": rec.get("exp_name"),
                    "dataset_name": dataset_name,
                    "source": rec.get("source"),
                    "algorithm": rec.get("algorithm") or result.get("algorithm_name"),
                    "seed": rec.get("seed"),
                    "args_id": rec.get("args_id"),
                    "output_root": rec.get("output_root"),
                }

                for k, v in rec.items():
                    if k in run:
                        continue
                    if isinstance(k, str) and k.endswith(("_acc_final", "_acc_best", "_loss_final", "_loss_best")):
                        run[k] = _to_float(v)

                for k, v in result.items():
                    if _is_scalar(v):
                        run[k] = v

                run_rows.append(run)

                err = rec.get("error")
                if isinstance(err, dict):
                    error_rows.append({
                        "file": str(path),
                        "line": line_no,
                        "run_id": rec.get("run_id"),
                        "algorithm": run.get("algorithm"),
                        "seed": run.get("seed"),
                        "error_type": err.get("type"),
                        "error_message": err.get("message"),
                    })

                history = result.get("history")
                if isinstance(history, list):
                    for event in history:
                        if not isinstance(event, dict):
                            continue
                        step = event.get("step")
                        train = event.get("train") if isinstance(event.get("train"), dict) else {}
                        eval_dict = event.get("eval") if isinstance(event.get("eval"), dict) else {}
                        for split, metric in eval_dict.items():
                            metric = metric if isinstance(metric, dict) else {}
                            history_rows.append({
                                "run_id": run.get("run_id"),
                                "algorithm": run.get("algorithm"),
                                "seed": run.get("seed"),
                                "exp_name": run.get("exp_name"),
                                "step": step,
                                "split": split,
                                "train_loss": _to_float(train.get("loss")),
                                "accuracy": _to_float(metric.get("accuracy")),
                                "macro_f1": _to_float(metric.get("macro_f1")),
                                "macro_recall": _to_float(metric.get("macro_recall")),
                            })

                for stage_key in ("iwildcam_metrics_best", "iwildcam_metrics_final"):
                    stage_payload = result.get(stage_key)
                    if not isinstance(stage_payload, dict):
                        continue
                    stage = stage_key.replace("iwildcam_metrics_", "")
                    for split, metric in stage_payload.items():
                        metric = metric if isinstance(metric, dict) else {}
                        split_rows.append({
                            "run_id": run.get("run_id"),
                            "algorithm": run.get("algorithm"),
                            "seed": run.get("seed"),
                            "exp_name": run.get("exp_name"),
                            "stage": stage,
                            "split": split,
                            "accuracy": _to_float(metric.get("accuracy")),
                            "macro_f1": _to_float(metric.get("macro_f1")),
                            "macro_recall": _to_float(metric.get("macro_recall")),
                            "alpha": _to_float(metric.get("alpha")),
                        })

                eval_metrics = result.get("metrics")
                if isinstance(eval_metrics, list):
                    eval_alpha = _to_float(result.get("eval_alpha"))
                    for metric in eval_metrics:
                        if not isinstance(metric, dict):
                            continue
                        eval_rows.append({
                            "run_id": run.get("run_id"),
                            "algorithm": run.get("algorithm"),
                            "seed": run.get("seed"),
                            "exp_name": run.get("exp_name"),
                            "split": metric.get("split"),
                            "accuracy": _to_float(metric.get("accuracy")),
                            "macro_f1": _to_float(metric.get("macro_f1")),
                            "macro_recall": _to_float(metric.get("macro_recall")),
                            "eval_alpha": eval_alpha if eval_alpha is not None else _to_float(metric.get("alpha")),
                        })

    runs_df = pd.DataFrame(run_rows)
    history_df = pd.DataFrame(history_rows)
    split_df = pd.DataFrame(split_rows)
    eval_df = pd.DataFrame(eval_rows)
    errors_df = pd.DataFrame(error_rows)
    return runs_df, history_df, split_df, eval_df, errors_df


In [ ]:
# Load all records for iWildCam
runs_df, history_df, split_df, eval_df, errors_df = load_iwildcam_records(RESULTS_ROOT, DATASET_FILTER)

print(f"runs rows: {len(runs_df):,}")
print(f"history rows: {len(history_df):,}")
print(f"split metric rows: {len(split_df):,}")
print(f"eval metric rows: {len(eval_df):,}")
print(f"error rows: {len(errors_df):,}")

if not runs_df.empty:
    display(runs_df.head(3))
if not errors_df.empty:
    print("Sample errors:")
    display(errors_df.head(5))


## Run Health and Coverage


In [ ]:
if runs_df.empty:
    print("No iWildCam records found under RESULTS_ROOT.")
else:
    health = (
        runs_df
        .groupby(["algorithm", "status"], dropna=False)
        .size()
        .rename("n")
        .reset_index()
        .sort_values(["algorithm", "status"])
    )
    display(health)

    coverage = (
        runs_df
        .pivot_table(index="algorithm", columns="seed", values="status", aggfunc="first")
        .sort_index()
    )
    display(coverage)


## Build Analysis View (Best-Val Metrics)

This merges either flat run columns (`val_acc_best`, `val_macro_f1_best`, ...) or the structured `iwildcam_metrics_best` payload.


In [ ]:
def build_best_val_view(runs_df: pd.DataFrame, split_df: pd.DataFrame) -> pd.DataFrame:
    base_cols = ["run_id", "algorithm", "seed", "exp_name", "status"]
    base = runs_df[base_cols].copy() if set(base_cols).issubset(runs_df.columns) else pd.DataFrame(columns=base_cols)

    for col in ["val_acc_best", "val_macro_f1_best", "val_macro_recall_best"]:
        if col in runs_df.columns:
            base[col] = pd.to_numeric(runs_df[col], errors="coerce")

    needs_merge = any(col not in base.columns for col in ["val_acc_best", "val_macro_f1_best", "val_macro_recall_best"])
    if needs_merge and not split_df.empty:
        best_val = split_df[(split_df["stage"] == "best") & (split_df["split"] == "val")].copy()
        best_val = best_val[["run_id", "accuracy", "macro_f1", "macro_recall"]].rename(
            columns={
                "accuracy": "val_acc_best",
                "macro_f1": "val_macro_f1_best",
                "macro_recall": "val_macro_recall_best",
            }
        )
        base = base.merge(best_val, on="run_id", how="left", suffixes=("", "_from_split"))
        for col in ["val_acc_best", "val_macro_f1_best", "val_macro_recall_best"]:
            alt = f"{col}_from_split"
            if alt in base.columns:
                base[col] = base[col].where(base[col].notna(), base[alt]) if col in base.columns else base[alt]
                base = base.drop(columns=[alt])

    return base


best_view = build_best_val_view(runs_df, split_df)
best_ok = best_view[best_view["status"] == "ok"].copy()

print(f"best_view rows: {len(best_view):,}")
print(f"best_ok rows: {len(best_ok):,}")

if not best_ok.empty:
    display(best_ok.sort_values(["algorithm", "seed"]).head(20))


## Algorithm Summary Table


In [ ]:
if best_ok.empty:
    print("No successful runs available for summary.")
else:
    summary = (
        best_ok
        .groupby("algorithm", dropna=False)
        .agg(
            n_runs=("run_id", "count"),
            n_seeds=("seed", "nunique"),
            val_acc_best_mean=("val_acc_best", "mean"),
            val_acc_best_std=("val_acc_best", "std"),
            val_f1_best_mean=("val_macro_f1_best", "mean"),
            val_f1_best_std=("val_macro_f1_best", "std"),
            val_recall_best_mean=("val_macro_recall_best", "mean"),
            val_recall_best_std=("val_macro_recall_best", "std"),
        )
        .reset_index()
        .sort_values("val_acc_best_mean", ascending=False)
    )
    display(summary)


## Visualization: Best Validation Metrics by Algorithm


In [ ]:
if best_ok.empty:
    print("No successful runs available for plotting.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    metric_specs = [
        ("val_acc_best", "Validation Accuracy (best)", axes[0]),
        ("val_macro_f1_best", "Validation Macro-F1 (best)", axes[1]),
    ]

    for metric, title, ax in metric_specs:
        data = best_ok[["algorithm", metric]].dropna().copy()
        stats = data.groupby("algorithm", as_index=False)[metric].agg(["mean", "std"]).reset_index()
        stats["std"] = stats["std"].fillna(0.0)

        palette = sns.color_palette("Set2", n_colors=len(stats))
        ax.bar(stats["algorithm"], stats["mean"], yerr=stats["std"], capsize=6, color=palette)
        ax.set_title(title)
        ax.set_xlabel("algorithm")
        ax.set_ylabel(metric)
        ax.tick_params(axis="x", rotation=25)

    plt.tight_layout()
    fig.savefig(FIG_DIR / "best_val_metrics_by_algorithm.png", dpi=150, bbox_inches="tight")
    plt.show()


## Visualization: Distribution Across Seeds


In [ ]:
if best_ok.empty:
    print("No successful runs available for box plots.")
else:
    melted = best_ok.melt(
        id_vars=["algorithm", "seed"],
        value_vars=["val_acc_best", "val_macro_f1_best"],
        var_name="metric",
        value_name="value",
    ).dropna()

    plt.figure(figsize=(12, 6))
    sns.boxplot(data=melted, x="algorithm", y="value", hue="metric", palette="tab10")
    sns.stripplot(data=melted, x="algorithm", y="value", hue="metric", dodge=True, alpha=0.45, color="black", size=3)

    handles, labels = plt.gca().get_legend_handles_labels()
    if len(labels) >= 2:
        plt.legend(handles[:2], labels[:2], title="metric", loc="best")

    plt.title("Seed-Level Metric Distributions")
    plt.xticks(rotation=25)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "seed_distributions_boxplot.png", dpi=150, bbox_inches="tight")
    plt.show()


## Training Dynamics from `result.history`


In [ ]:
if history_df.empty:
    print("No history rows found. This can happen if runs failed or used eval-only records.")
else:
    val_hist = history_df[history_df["split"] == "val"].copy()
    if val_hist.empty:
        print("No 'val' split rows found in history.")
    else:
        val_hist["step"] = pd.to_numeric(val_hist["step"], errors="coerce")
        val_hist = val_hist.dropna(subset=["step"]).sort_values(["algorithm", "seed", "step"])

        agg = (
            val_hist
            .groupby(["algorithm", "step"], as_index=False)
            .agg(
                acc_mean=("accuracy", "mean"),
                acc_std=("accuracy", "std"),
                f1_mean=("macro_f1", "mean"),
                f1_std=("macro_f1", "std"),
            )
        )
        agg[["acc_std", "f1_std"]] = agg[["acc_std", "f1_std"]].fillna(0.0)

        fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharex=True)
        palette = sns.color_palette("tab10", n_colors=max(1, agg["algorithm"].nunique()))
        color_map = {alg: palette[i % len(palette)] for i, alg in enumerate(sorted(agg["algorithm"].dropna().unique()))}

        for alg in sorted(agg["algorithm"].dropna().unique()):
            g = agg[agg["algorithm"] == alg].sort_values("step")
            c = color_map[alg]

            axes[0].plot(g["step"], g["acc_mean"], label=alg, color=c, linewidth=2.5)
            axes[0].fill_between(g["step"], g["acc_mean"] - g["acc_std"], g["acc_mean"] + g["acc_std"], alpha=0.2, color=c)

            axes[1].plot(g["step"], g["f1_mean"], label=alg, color=c, linewidth=2.5)
            axes[1].fill_between(g["step"], g["f1_mean"] - g["f1_std"], g["f1_mean"] + g["f1_std"], alpha=0.2, color=c)

        axes[0].set_title("Validation Accuracy vs Step (mean ± std)")
        axes[1].set_title("Validation Macro-F1 vs Step (mean ± std)")
        axes[0].set_xlabel("step"); axes[1].set_xlabel("step")
        axes[0].set_ylabel("accuracy"); axes[1].set_ylabel("macro_f1")
        axes[0].legend(); axes[1].legend()

        plt.tight_layout()
        fig.savefig(FIG_DIR / "val_training_dynamics.png", dpi=150, bbox_inches="tight")
        plt.show()


## Optional: IRO Alpha-Preference Sweep (`iro eval` outputs)

This section works when evaluation records exist with `result.metrics` and `result.eval_alpha`.


In [ ]:
if eval_df.empty:
    print("No eval-mode metrics detected yet.")
    print("Run iro eval with multiple eval.alpha values, then reload this notebook.")
else:
    d = eval_df[(eval_df["algorithm"] == "iro") & (eval_df["split"] == "val")].copy()
    d["eval_alpha"] = pd.to_numeric(d["eval_alpha"], errors="coerce")
    d = d.dropna(subset=["eval_alpha"]) 

    if d.empty:
        print("No IRO val-split eval records with eval_alpha found.")
    else:
        alpha_curve = (
            d
            .groupby("eval_alpha", as_index=False)
            .agg(
                acc_mean=("accuracy", "mean"),
                acc_std=("accuracy", "std"),
                f1_mean=("macro_f1", "mean"),
                f1_std=("macro_f1", "std"),
            )
            .sort_values("eval_alpha")
        )
        alpha_curve[["acc_std", "f1_std"]] = alpha_curve[["acc_std", "f1_std"]].fillna(0.0)

        display(alpha_curve)

        fig, ax = plt.subplots(figsize=(10, 6))
        ax.plot(alpha_curve["eval_alpha"], alpha_curve["acc_mean"], marker="o", linewidth=2.5, label="val accuracy")
        ax.fill_between(alpha_curve["eval_alpha"], alpha_curve["acc_mean"] - alpha_curve["acc_std"], alpha_curve["acc_mean"] + alpha_curve["acc_std"], alpha=0.2)

        ax.plot(alpha_curve["eval_alpha"], alpha_curve["f1_mean"], marker="s", linewidth=2.5, label="val macro_f1")
        ax.fill_between(alpha_curve["eval_alpha"], alpha_curve["f1_mean"] - alpha_curve["f1_std"], alpha_curve["f1_mean"] + alpha_curve["f1_std"], alpha=0.2)

        ax.set_title("IRO Preference Sweep on iWildCam (val split)")
        ax.set_xlabel("eval.alpha")
        ax.set_ylabel("metric")
        ax.legend()
        ax.grid(alpha=0.25)

        plt.tight_layout()
        fig.savefig(FIG_DIR / "iro_alpha_sweep_val.png", dpi=150, bbox_inches="tight")
        plt.show()


## Export Tables


In [ ]:
runs_csv = OUT_DIR / "iwildcam_runs_flat.csv"
history_csv = OUT_DIR / "iwildcam_history_flat.csv"
split_csv = OUT_DIR / "iwildcam_split_metrics_flat.csv"
eval_csv = OUT_DIR / "iwildcam_eval_metrics_flat.csv"
errors_csv = OUT_DIR / "iwildcam_errors_flat.csv"

runs_df.to_csv(runs_csv, index=False)
history_df.to_csv(history_csv, index=False)
split_df.to_csv(split_csv, index=False)
eval_df.to_csv(eval_csv, index=False)
errors_df.to_csv(errors_csv, index=False)

print("Wrote:")
print(" -", runs_csv)
print(" -", history_csv)
print(" -", split_csv)
print(" -", eval_csv)
print(" -", errors_csv)
print("Figure directory:", FIG_DIR)


## Notes

- For model-selection comparisons, use `*_best` metrics.
- For "what the final checkpoint does", use `*_final` metrics.
- iWildCam metrics can be low early in training; compare across equal step budgets.
- For preference analysis, sweep `eval.alpha` on fixed IRO checkpoints and re-run this notebook.
